# Семинар. Обучение линейных моделей и отбор признаков



Мы поработаем с данными о сообществах в США. Будем предсказывать количество насильственных преступлений относительно численности населения.

[Описание датасета](http://archive.ics.uci.edu/ml/datasets/communities+and+crime)
[Датасет на кэггле](https://www.kaggle.com/kkanda/communities%20and%20crime%20unnormalized%20data%20set)

In [1]:
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import pandas as pd

In [2]:
CRIMES = "https://raw.githubusercontent.com/evgpat/datasets/main/crimedata.csv"

In [3]:
data = pd.read_csv(CRIMES, na_values=["?"])

# оставим лишь нужные колонки
requiredColumns = [5, 6] + list(range(11, 26)) + list(range(32, 103)) + [145]
data = data[data.columns[requiredColumns]]

# некоторые значения целевой переменной пропущены
X = data.loc[data["ViolentCrimesPerPop"].notnull(), :].drop(
    "ViolentCrimesPerPop", axis=1
)
y = data["ViolentCrimesPerPop"][X.index]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0, test_size=0.2)

### Baseline
Обучим линейную регрессию и выведем качество по метрике MSE на обучающей и тестовой выборке.

In [4]:
lr = LinearRegression().fit(X_train, y_train)
print(f"Train: {mean_squared_error(y_train, lr.predict(X_train))}")
print(f"Test: {mean_squared_error(y_test, lr.predict(X_test))}")

Train: 118390.69446924044
Test: 229491.85262710578


### Scaling
Попробуем MinMaxScaler.

MinMaxScaler:

$$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

StandardScaler:

$$z = \frac{x - \mu}{\sigma}
$$ где

- x — исходное значение
- μ — среднее по признаку
- σ — стандартное отклонение
- z — нормализованное значение

In [5]:
sc = MinMaxScaler()
X_train_scaled = pd.DataFrame(data=sc.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(data=sc.transform(X_test), columns=X_test.columns)

In [6]:
# sc_1 = StandardScaler()
# X_train_scaled = pd.DataFrame(data=sc_1.fit_transform(X_train), columns=X_train.columns)
# X_test_scaled = pd.DataFrame(data=sc_1.transform(X_test), columns=X_test.columns)

**Задание.** Напишите код обучения линейной регрессии на масштабированных признаках и выведите ошибку на обучающей и тестовой выборках.

In [7]:
lr = LinearRegression().fit(X_train_scaled, y_train)
print(f"Train: {mean_squared_error(y_train, lr.predict(X_train_scaled))}")
print(f"Test: {mean_squared_error(y_test, lr.predict(X_test_scaled))}")

Train: 118390.69446924046
Test: 229491.85261381435


### Отбор признаков на основе дисперсии

Полезны ли признаки, имеющие высокую дисперсию? А низкую?

In [8]:
features_variance = X_train_scaled.var().sort_values(ascending=False)
features_variance.head()

,0
pctUrban,0.198050
RentHighQ,0.062216
MedYrHousBuilt,0.054286
OwnOccHiQuart,0.047559
MedRent,0.046091


Попробуем удалить признаки с самой низкой дисперсией и посмотреть, как изменится качество. В sklearn есть специальный инструмент для такого наивного отбора признаков. Стоит ли нормализовать перед этим признаки?

In [9]:
from sklearn.feature_selection import VarianceThreshold

In [10]:
# можно убрать все признаки, дисперсия которых меньше заданного значения
vs_transformer = VarianceThreshold(0.01)

X_train_var = pd.DataFrame(
    data=vs_transformer.fit_transform(X_train_scaled),
    columns=X_train_scaled.columns[vs_transformer.get_support()],
)
X_test_var = pd.DataFrame(
    data=vs_transformer.transform(X_test_scaled),
    columns=X_test_scaled.columns[vs_transformer.get_support()],
)

X_train_var.shape

(1595, 73)

In [11]:
lr = LinearRegression().fit(X_train_var, y_train)
print(f"Train: {mean_squared_error(y_train, lr.predict(X_train_var))}")
print(f"Test: {mean_squared_error(y_test, lr.predict(X_test_var))}")

Train: 125460.03620873159
Test: 157285.6218456544


### Отбор признаков на основе корреляции с целевой переменной
Можно выбрать k признаков, которые дают наиболее высокие значения корреляции с целевой переменной.

In [12]:
from sklearn.feature_selection import SelectKBest, f_regression

In [13]:
# Выбираем 15 лучших признаков
sb = SelectKBest(f_regression, k=15)

X_train_kbest = pd.DataFrame(
    data=sb.fit_transform(X_train_var, y_train),
    columns=X_train_var.columns[sb.get_support()],
)
X_test_kbest = pd.DataFrame(
    data=sb.transform(X_test_var), columns=X_test_var.columns[sb.get_support()]
)

In [14]:
lr = LinearRegression().fit(X_train_kbest, y_train)
print(f"Train: {mean_squared_error(y_train, lr.predict(X_train_kbest))}")
print(f"Test: {mean_squared_error(y_test, lr.predict(X_test_kbest))}")

Train: 146478.7820172876
Test: 160372.23523784862


А можно выбрать самые значимые признаки с точки зрения регрессии с $L_1$-регуляризацией.

In [15]:
from sklearn.feature_selection import SelectFromModel

In [16]:
lasso = Lasso(5.0)
l1_select = SelectFromModel(lasso)

X_train_l1 = pd.DataFrame(
    data=l1_select.fit_transform(X_train_var, y_train),
    columns=X_train_var.columns[l1_select.get_support()],
)
X_test_l1 = pd.DataFrame(
    data=l1_select.transform(X_test_var),
    columns=X_test_var.columns[l1_select.get_support()],
)

X_train_l1.shape

(1595, 10)

In [17]:
lr = LinearRegression().fit(X_train_l1, y_train)
print(f"Train: {mean_squared_error(y_train, lr.predict(X_train_l1))}")
print(f"Test: {mean_squared_error(y_test, lr.predict(X_test_l1))}")

Train: 140633.82259967248
Test: 161202.08743230256


### Зададим все преобразования, отбор признаков и обучение при помощи Pipeline

In [18]:
from sklearn.pipeline import Pipeline

pipe = Pipeline(
    steps=[
        ("scaler", MinMaxScaler()), #Standard
        ("variance", VarianceThreshold(0.01)),
        ("selection", SelectFromModel(Lasso(5.0))),
        ("regressor", LinearRegression()),
    ]
)

pipe.fit(X_train, y_train)

pipe.named_steps

{'scaler': MinMaxScaler(),
 'variance': VarianceThreshold(threshold=0.01),
 'selection': SelectFromModel(estimator=Lasso(alpha=5.0)),
 'regressor': LinearRegression()}

In [19]:
print(f"Train: {mean_squared_error(y_train, pipe.predict(X_train))}")
print(f"Test: {mean_squared_error(y_test, pipe.predict(X_test))}")

Train: 140633.82259967248
Test: 161202.08743230256


Можно даже перебрать гиперпараметры с помощью `GridSearch`:

In [20]:
pipe.get_params()

{'memory': None,
 'steps': [('scaler', MinMaxScaler()),
  ('variance', VarianceThreshold(threshold=0.01)),
  ('selection', SelectFromModel(estimator=Lasso(alpha=5.0))),
  ('regressor', LinearRegression())],
 'transform_input': None,
 'verbose': False,
 'scaler': MinMaxScaler(),
 'variance': VarianceThreshold(threshold=0.01),
 'selection': SelectFromModel(estimator=Lasso(alpha=5.0)),
 'regressor': LinearRegression(),
 'scaler__clip': False,
 'scaler__copy': True,
 'scaler__feature_range': (0, 1),
 'variance__threshold': 0.01,
 'selection__estimator__alpha': 5.0,
 'selection__estimator__copy_X': True,
 'selection__estimator__fit_intercept': True,
 'selection__estimator__max_iter': 1000,
 'selection__estimator__positive': False,
 'selection__estimator__precompute': False,
 'selection__estimator__random_state': None,
 'selection__estimator__selection': 'cyclic',
 'selection__estimator__tol': 0.0001,
 'selection__estimator__warm_start': False,
 'selection__estimator': Lasso(alpha=5.0),
 'se

In [21]:
param_grid = {
    "variance__threshold": [0.005, 0.0075, 0.009, 0.01, 0.011, 0.012],
    "selection__estimator__alpha": [0.5, 1.0, 1.5, 2.0, 5.0, 10.0]
}
grid_search = GridSearchCV(pipe, param_grid, cv=5, verbose=True)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 36 candidates, totalling 180 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', MinMaxScaler()),
                                       ('variance',
                                        VarianceThreshold(threshold=0.01)),
                                       ('selection',
                                        SelectFromModel(estimator=Lasso(alpha=5.0))),
                                       ('regressor', LinearRegression())]),
             param_grid={'selection__estimator__alpha': [0.5, 1.0, 1.5, 2.0,
                                                         5.0, 10.0],
                         'variance__threshold': [0.005, 0.0075, 0.009, 0.01,
                                                 0.011, 0.012]},
             verbose=True)

In [22]:
pipe_best = grid_search.best_estimator_
pipe_best.named_steps

{'scaler': MinMaxScaler(),
 'variance': VarianceThreshold(threshold=0.011),
 'selection': SelectFromModel(estimator=Lasso(alpha=2.0)),
 'regressor': LinearRegression()}

In [23]:
pipe_best.fit(X_train, y_train)
print(f"Train: {mean_squared_error(y_train, pipe_best.predict(X_train))}")
print(f"Test: {mean_squared_error(y_test, pipe_best.predict(X_test))}")

Train: 135289.6306759399
Test: 155343.56080298405
